# Vistazo a bronze: tablas que no están en silver

Lee los CSV crudos del espejo local de bronze (`data/bronze`, ver `make mirror`) con DuckDB, para mirar las tablas que todavía no pasan a silver (hoy solo `digital_events`, excluida por tamaño en `transform/generate_silver.py`).

Diferencias con silver (`query_silver.ipynb`):
- **Todo es texto** (`all_varchar`): sin tipos, sin mapeos de valores en español, sin deduplicación ni `_dq_issues`. Para números y fechas hay que castear (`try_cast`).
- **Más lento**: cada consulta recorre los CSV (para `digital_events`, 1.097 archivos y 3,7 GB).
- Problemas conocidos de los datos: `docs/known_issues.md`.

In [1]:
import time
from pathlib import Path

import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
BRONZE = ROOT / "data/bronze/data"
WAREHOUSE = ROOT / "data/warehouse.duckdb"

con = duckdb.connect()  # en memoria; bronze se lee directo de los CSV


def bronze_path(table: str) -> str:
    """Glob de los CSV de una tabla: un archivo (dimensiones) o particiones diarias (hechos)."""
    single = BRONZE / f"{table}.csv"
    return str(single) if single.exists() else str(BRONZE / table / "*/*/*/*.csv")

## ¿Qué tablas faltan en silver?

In [2]:
bronze_tables = sorted(p.stem for p in BRONZE.iterdir() if p.suffix == ".csv" or p.is_dir())

silver_tables = []
if WAREHOUSE.exists():
    con.sql(f"attach '{WAREHOUSE}' as silver (read_only)")
    silver_tables = [r[0] for r in con.sql(
        "select table_name from duckdb_tables() where database_name = 'silver'").fetchall()]

missing = [t for t in bronze_tables if t not in silver_tables]
print("en bronze:", len(bronze_tables), "| en silver:", len([t for t in bronze_tables if t in silver_tables]))
print("solo en bronze:", missing)

en bronze: 13 | en silver: 12
solo en bronze: ['digital_events']


In [3]:
# Una vista por tabla faltante: se consultan como tablas normales (from digital_events ...)
for t in missing:
    con.sql(f"""
        create or replace view {t} as
        select * from read_csv('{bronze_path(t)}', header = true, all_varchar = true,
                               filename = true, hive_partitioning = true)
    """)
    print(f"vista '{t}' → {bronze_path(t).replace(str(ROOT) + '/', '')}")

vista 'digital_events' → data/bronze/data/digital_events/*/*/*/*.csv


## `digital_events`

Eventos de canales digitales (app y web). Según el diccionario: 10 M de filas; en los datos hay 15,6 M (ver `docs/known_issues.md`).

In [4]:
con.sql("describe digital_events").show(max_rows=40)

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ event_id         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ event_date       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ process_date     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ session_id       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ event_type       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ event_category   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ channel          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ platform         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ browser          │ VARC

In [5]:
# Recorre los 1.097 archivos: la consulta más cara de la notebook
t0 = time.time()
con.sql("""
    select year, count(*) as filas, min(event_date) as desde, max(event_date) as hasta
    from digital_events
    group by 1
    order by 1
""").show()
print(f"{time.time() - t0:.1f} s")

┌───────┬─────────┬─────────────────────┬─────────────────────┐
│ year  │  filas  │        desde        │        hasta        │
│ int64 │  int64  │       varchar       │       varchar       │
├───────┼─────────┼─────────────────────┼─────────────────────┤
│  2023 │ 2926425 │ 2023-06-17 06:02:03 │ 2024-01-01 06:02:16 │
│  2024 │ 5057968 │ 2024-01-01 06:00:27 │ 2025-01-01 06:07:38 │
│  2025 │ 5173832 │ 2025-01-01 06:00:27 │ 2026-01-01 06:01:50 │
│  2026 │ 2462769 │ 2026-01-01 06:00:17 │ 2026-06-18 06:04:05 │
└───────┴─────────┴─────────────────────┴─────────────────────┘

2.0 s


In [6]:
# Muestra de filas (solo lee los primeros archivos)
con.sql("select * exclude (filename) from digital_events limit 5").df()

,event_id,event_date,process_date,customer_id,session_id,event_type,event_category,channel,platform,browser,...,ip_country,ip_city,is_mobile,referrer,utm_source,utm_medium,utm_campaign,day,month,year
0,EVT-ZUVDGU5COPO2MJ8G9XO7,2023-06-17 08:52:08,2023-06-17,CLI-X4M6ICRGTT9Z,SES-8QOW3F17I07NJ7LT0TPC5YC33ULT,Login,Authentication,Desktop Web,Linux,Chrome,...,Colombia,Medellín,False,None,None,None,None,17,06,2023
1,EVT-Z4HRU55OLDLG9JXO4VRR,2023-06-17 08:52:47,2023-06-17,CLI-X4M6ICRGTT9Z,SES-8QOW3F17I07NJ7LT0TPC5YC33ULT,PageView,Transaction,Desktop Web,Linux,Chrome,...,Colombia,Medellín,False,None,None,None,None,17,06,2023
2,EVT-4DZT88K2QTZJ4NY5ILYX,2023-06-17 08:53:25,2023-06-17,CLI-X4M6ICRGTT9Z,SES-8QOW3F17I07NJ7LT0TPC5YC33ULT,Login,Authentication,Desktop Web,Linux,Chrome,...,Colombia,Medellín,False,None,None,None,None,17,06,2023
3,EVT-HJQVK8HE3NCEV1NN2QS3,2023-06-17 08:53:32,2023-06-17,CLI-X4M6ICRGTT9Z,SES-8QOW3F17I07NJ7LT0TPC5YC33ULT,Error,Transaction,Desktop Web,Linux,Chrome,...,Colombia,NaN,False,None,None,None,None,17,06,2023
4,EVT-DZ9SGFYZKP5AYKJQ2LIU,2023-06-17 08:54:07,2023-06-17,CLI-X4M6ICRGTT9Z,SES-8QOW3F17I07NJ7LT0TPC5YC33ULT,Logout,Authentication,Desktop Web,Linux,Chrome,...,Colombia,Medellín,False,None,None,None,None,17,06,2023


### Distribuciones

Para iterar rápido, las siguientes celdas usan un mes (`SAMPLE_MONTH`). Cambiar a `None` para recorrer todo.

In [7]:
SAMPLE_MONTH = ("2025", "01")  # (year, month) o None para toda la tabla

where = f"where year = '{SAMPLE_MONTH[0]}' and month = '{SAMPLE_MONTH[1]}'" if SAMPLE_MONTH else ""
con.sql(f"create or replace temp table de_sample as select * exclude (filename) from digital_events {where}")
print(con.sql("select count(*) from de_sample").fetchone()[0], "filas en la muestra")

443948 filas en la muestra


In [8]:
for col in ["event_type", "event_category", "channel", "platform", "is_mobile"]:
    con.sql(f"""
        select {col}, count(*) as n, round(100 * count(*) / sum(count(*)) over (), 1) as pct
        from de_sample
        group by 1
        order by n desc
    """).show()

┌────────────┬────────┬────────┐
│ event_type │   n    │  pct   │
│  varchar   │ int64  │ double │
├────────────┼────────┼────────┤
│ PageView   │ 169921 │   38.3 │
│ Click      │ 101644 │   22.9 │
│ Login      │  69300 │   15.6 │
│ Logout     │  69284 │   15.6 │
│ FormSubmit │  16794 │    3.8 │
│ Error      │  10121 │    2.3 │
│ Purchase   │   6884 │    1.6 │
└────────────┴────────┴────────┘

┌────────────────┬────────┬────────┐
│ event_category │   n    │  pct   │
│    varchar     │ int64  │ double │
├────────────────┼────────┼────────┤
│ Authentication │ 138584 │   31.2 │
│ Navigation     │ 112459 │   25.3 │
│ Product        │ 107337 │   24.2 │
│ Transaction    │  85568 │   19.3 │
└────────────────┴────────┴────────┘

┌─────────────┬────────┬────────┐
│   channel   │   n    │  pct   │
│   varchar   │ int64  │ double │
├─────────────┼────────┼────────┤
│ Android App │ 155727 │   35.1 │
│ iOS App     │ 110720 │   24.9 │
│ Mobile Web  │  89255 │   20.1 │
│ Desktop Web │  88246 │   19.9

In [9]:
# Nulos por columna (en la muestra)
cols = [r[0] for r in con.sql("describe de_sample").fetchall() if r[0] not in ("year", "month", "day")]
con.sql("select " + ", ".join(
    f"round(100 * avg(({c} is null)::int), 1) as \"{c}\"" for c in cols
) + " from de_sample").df().T.rename(columns={0: "null_%"}).sort_values("null_%", ascending=False)

,null_%
event_value,94.9
utm_campaign,94.6
utm_medium,94.6
utm_source,94.6
referrer,93.3
product_id,90.7
duration_seconds,63.7
browser,62.0
app_version,43.0
ip_city,27.6


In [10]:
# Montos y duraciones (texto → número con try_cast)
con.sql("""
    select
        count(try_cast(event_value as decimal(15,2))) as con_valor,
        round(avg(try_cast(event_value as decimal(15,2))), 2) as valor_prom,
        round(median(try_cast(duration_seconds as double)), 1) as duracion_mediana_s,
        count(distinct customer_id) as clientes,
        count(distinct session_id) as sesiones
    from de_sample
""").show()

┌───────────┬────────────┬────────────────────┬──────────┬──────────┐
│ con_valor │ valor_prom │ duracion_mediana_s │ clientes │ sesiones │
│   int64   │   double   │       double       │  int64   │  int64   │
├───────────┼────────────┼────────────────────┼──────────┼──────────┤
│     22490 │    2516.29 │              152.0 │    36588 │    52303 │
└───────────┴────────────┴────────────────────┴──────────┴──────────┘



In [11]:
# Eventos que referencian productos: ¿se pueden cruzar con silver?
if silver_tables:
    con.sql("""
        select d.event_type, p.product_type, count(*) as n
        from de_sample d
        join silver.products p on p.product_id = d.product_id
        group by all
        order by n desc
        limit 15
    """).show()

┌────────────┬──────────────────┬───────┐
│ event_type │   product_type   │   n   │
│  varchar   │     varchar      │ int64 │
├────────────┼──────────────────┼───────┤
│ PageView   │ Savings Account  │  6550 │
│ Click      │ Savings Account  │  5892 │
│ PageView   │ Credit Card      │  5453 │
│ PageView   │ Checking Account │  5389 │
│ Click      │ Checking Account │  4877 │
│ Click      │ Credit Card      │  4814 │
│ PageView   │ Debit Card       │  2219 │
│ Click      │ Debit Card       │  1964 │
│ PageView   │ Personal Loan    │  1020 │
│ Click      │ Personal Loan    │   946 │
│ PageView   │ Mortgage         │   664 │
│ Click      │ Mortgage         │   559 │
│ PageView   │ Investment       │   317 │
│ Click      │ Investment       │   285 │
│ Click      │ Insurance        │    96 │
└────────────┴──────────────────┴───────┘
  15 rows                     3 columns



## Consultar cualquier tabla de bronze

También las que sí están en silver, para comparar el dato crudo con el limpio.

In [12]:
def bronze(table: str):
    """Relación DuckDB sobre los CSV crudos de una tabla de bronze."""
    return con.sql(f"select * from read_csv('{bronze_path(table)}', header = true, all_varchar = true)")


bronze("branches").limit(3).show()

┌──────────────┬─────────────┬────────────────────────────────┬─────────────┬──────────────────────────────────┬─────────────┬─────────────────┬─────────┬─────────────┬─────────────────┬──────────────────┬─────────────────────────────┬──────────────┬──────────────┬──────────┬───────────┬────────────────────┬─────────────────────┬────────────┬──────────────┬─────────────────────┬───────────────┐
│  branch_id   │ branch_code │          branch_name           │ branch_type │             address              │    city     │      state      │ country │ postal_code │ geographic_zone │      phone       │            email            │ opening_time │ closing_time │ has_atms │ atm_count │ has_teller_windows │ teller_window_count │  latitude  │  longitude   │ branch_opening_date │ branch_status │
│   varchar    │   varchar   │            varchar             │   varchar   │             varchar              │   varchar   │     varchar     │ varchar │   varchar   │     varchar     │     varchar      